In [ ]:
# ---- 0. 环境设置 -----------------------------------------------------------
Sys.setenv(SP_EVOLUTION_STATUS = "2")

set.seed(1)
library(SeuratObject)
library(Matrix)
library(dplyr)
library(tidyr)
library(readr)
library(DESeq2)

In [ ]:
# ---- 0. 参数设置 ------------------------------------------------------------
date <- "2026-07-06_2"
data_dir <- "/project/work/2026-07-06_development_DEG/Data/"
min_cells <- 30
max_cell <- Inf

celltypes <- c('Rod','Cone','AC','BC','HC','MG','RGC','RPC')

In [ ]:
# ---- 6. DESeq2 区域差异分析 (macula vs peripheral) --------------------------
cat("\n=== 6. DESeq2 区域差异分析 ===\n")

In [ ]:
for (ct in celltypes) {
  cat("  处理 celltype:", ct, "\n")
  
  pb_file <- paste0(data_dir, "2026-07-06_1", "_", ct, "_regional_pseudobulk.rds")
  if (!file.exists(pb_file)) next
  
  pb <- readRDS(pb_file)
  countData_all <- as.matrix(pb$counts)
  colData_all <- read.csv(paste0(data_dir, "2026-07-06_1", "_", ct, "_regional_colData.csv"), row.names = 1)
  colData_all$group <- rownames(colData_all)
  
  # 按 stage 分别分析
  for (st in c("Early", "Mid", "Late")) {
    cat("    Stage:", st, "\n")
    
    # 1. 筛选该 stage 的样本
    colData <- colData_all %>% filter(stage == st)
    
    # 2. 检查该 stage 内是否有配对 donor（同时有 macula + peripheral）
    donor_tissue <- colData %>% 
     dplyr::group_by(donor_age, tissue) %>% 
     dplyr::summarise(n = dplyr::n(), .groups = "drop")

    valid_donors <- donor_tissue %>% 
      dplyr::count(donor_age) %>% 
      dplyr::filter(n == 2) %>% 
      dplyr::pull(donor_age)
    
    if (length(valid_donors) < 2) {
      cat("      警告: ", st, " 有效配对 donor 不足, 跳过\n")
      next
    }
    
    # 3. 保留配对 donor
    colData <- colData %>% filter(donor_age %in% valid_donors)
    countData <- countData_all[, colData$group]
    
    # 4. 过滤低表达基因
    keep <- rowSums(countData) >= 10
    countData <- countData[keep, ]
    
    if (nrow(countData) == 0 || ncol(countData) == 0) next
    
    # 5. 在该 stage 内运行 DESeq2
    dds <- DESeqDataSetFromMatrix(
      countData = countData,
      colData = colData,
      design = ~ donor_age + tissue
    )
    
    dds <- DESeq(dds)
    res <- results(dds, contrast = c("tissue", "macula", "peripheral"))
    
    res_df <- as.data.frame(res) %>%
      mutate(gene = rownames(.)) %>%
      relocate(gene) %>%
      mutate(celltype = ct, stage = st, .before = 1)
    
    # 6. 保存结果
    write.csv(
      res_df, 
      paste0(data_dir, date, "_", ct, "_", st, "_macula_vs_peripheral.csv"), 
      row.names = FALSE
    )
    print(paste0(data_dir, date, "_", ct, "_", st, "_macula_vs_peripheral.csv"))
    cat("      完成: ", nrow(res_df), " 个基因\n")
  }
}

In [ ]:
# ---- 8. 合并 Regional DEGs --------------------------------------------------
cat("\n========================================\n")
cat("=== 8. 提取严格差异基因 + 合并结果 ===\n")
cat("========================================\n")

library(tidyverse)

padj_cutoff <- 0.05
logfc_cutoff <- 0.5

In [ ]:
# ==================== Regional DEGs 合并 ====================
cat("=== 处理 Regional DEGs ===\n")

reg_files <- list.files(data_dir, pattern = paste0("2026-07-06_2", "_.*_macula_vs_peripheral\\.csv$"), full.names = TRUE)

In [ ]:
if (length(reg_files) == 0) {
  cat("  警告: 未找到 regional DEG 文件\n")
} else {
  reg_list <- lapply(reg_files, function(f) {
    cat("  读取:", basename(f), "\n")
    df <- read.csv(f)
    
    core <- gsub(paste0("^", date, "_"), "", gsub("\\.csv$", "", basename(f)))
    parts <- strsplit(core, "_")[[1]]
    n <- length(parts)
    
    st <- parts[n - 3]
    ct <- paste(parts[1:(n - 4)], collapse = "_")
    
    df_filt <- df %>%
      filter(
        !is.na(padj), 
        padj < padj_cutoff, 
        !is.na(log2FoldChange), 
        abs(log2FoldChange) > logfc_cutoff
      ) %>%
      mutate(
        celltype = ct,
        stage = st,
        pair = "macula_vs_peripheral",
        direction = ifelse(log2FoldChange > 0, "macula_up", "peripheral_up")
      )
    
    return(df_filt)
  })
  
  regional_merged <- bind_rows(reg_list)
  out_reg <- paste0(data_dir, date, "_regional_degs_filtered_merged.csv")
  write.csv(regional_merged, out_reg, row.names = FALSE)
  cat("  完成: 合并", nrow(regional_merged), "条记录 ->", out_reg, "\n")
}

In [ ]:
regional_merged$celltype_stage <- paste0(regional_merged$celltype,"_",regional_merged$stage)